# Badminton Analysis Pipeline (Colab)
Runs the stabilized multi-trajectory pipeline on your hand-held phone match videos.

**Before running:** `Runtime ▸ Change runtime type ▸ GPU` (T4). Upload your video and 4 court corners.

## 1. Clone + install

In [ ]:
!git clone https://github.com/sujkuttan/baddyAnalysis.git
%cd baddyAnalysis
!pip install -r requirements.txt

## 2. Upload your match video

In [ ]:
from google.colab import files
uploaded = files.upload()   # select match.mp4
video_name = list(uploaded.keys())[0]
print('video:', video_name)

## 3. Pick 4 court corners
The first frame is shown. Edit `corners` below to the 4 court corners in order:
**TL, TR, BR, BL** (x,y in pixels, from the image).

In [ ]:
import cv2, os, json
from IPython.display import Image
os.makedirs('data', exist_ok=True)
cap = cv2.VideoCapture(video_name); ret, frame = cap.read(); cap.release()
cv2.imwrite('data/first_frame.jpg', frame)
display(Image('data/first_frame.jpg'))

# EDIT THESE using the image above (TL, TR, BR, BL)
corners = [
    [100, 100],
    [500, 100],
    [500, 900],
    [100, 900],
]
json.dump({'corners': corners}, open('corners.json','w'))
print('saved corners.json')

## 4. (Optional) TrackNet weights
Shuttle/contact detection needs `weights/TrackNet_best.pt`.
If you don't have it, the pipeline still runs (shuttle/contact will be degraded).
Upload it here, or place it in `weights/`.

In [ ]:
import os
os.makedirs('weights', exist_ok=True)
# files.upload()  # uncomment and upload TrackNet_best.pt into weights/
tracknet = 'weights/TrackNet_best.pt' if os.path.exists('weights/TrackNet_best.pt') else None
print('tracknet:', tracknet)

## 5. Run the pipeline
Produces `data/new_predictions.csv`, `data/metrics.json`, `data/annotated.mp4`,
`data/coverage_heatmap.png`, `data/fatigue.png`, `data/coaching_report.md`.
If `labels_import.csv` is present it fine-tunes the fusion classifier on your labeled shots.

In [ ]:
from src import pipeline
corners = json.load(open('corners.json'))['corners']
res = pipeline.run_full_pipeline(
    video_name, corners, out_dir='data',
    labels_csv='labels_import.csv', device='cuda',
    tracknet_weights=tracknet,
)
print('predictions:', res['predictions_csv'])
print('metrics:', res['metrics'])

## 6. A/B vs your BST pipeline
Export your BST predictions to `bst_predictions.csv` (columns `frame,predicted_stroke`),
upload it, then compare. The `labeled` rows in `labels_import.csv` are the shared ground truth.

In [ ]:
from google.colab import files
# files.upload()  # upload bst_predictions.csv (optional)
from src import pipeline
pipeline.ab_compare('labels_import.csv', 'data/new_predictions.csv', 'bst_predictions.csv')